# Train Crime-Event Prediction Model

Train the crime-event prediction model that powers the daily-options markets in the crimex app.
Trains a LightGBM regressor to predict the count of incidents in a (city, incident_type, horizon) window.
Exports a JSON tree-dump that the Node side (`lib/predictions/infrastructure/models/trained.ts`) can
evaluate without any ML runtime dependencies.

In [ ]:
%pip install -q lightgbm pandas numpy requests scikit-learn matplotlib

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import json
import math
import os
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import requests
import lightgbm as lgb
import matplotlib.pyplot as plt
from scipy.stats import poisson as scipy_poisson
from sklearn.metrics import log_loss as sk_log_loss
from sklearn.preprocessing import LabelEncoder

# ── Config — edit these before running ───────────────────────────────────────
ARCGIS_URL = "https://services2.arcgis.com/o1LYr96CpFkfsDJS/arcgis/rest/services/Crime_Map/FeatureServer/0"
LOOKBACK_DAYS = 730
HORIZONS_HOURS = [4, 8, 12, 24]
SUPABASE_URL = ""
SUPABASE_SERVICE_ROLE_KEY = ""
MODEL_ID = "trained-v1"
TIMEZONE = "America/Toronto"

In [ ]:
def fetch_arcgis(lookback_days: int) -> pd.DataFrame:
    """Paginated fetch from ArcGIS FeatureServer. Returns a flat DataFrame."""
    cutoff_ms = int(
        (datetime.now(timezone.utc).timestamp() - lookback_days * 86400) * 1000
    )
    records = []
    offset = 0
    page_size = 2000

    while True:
        params = {
            "where": "1=1",
            "outFields": "*",
            "f": "geojson",
            "resultOffset": offset,
            "resultRecordCount": page_size,
        }
        resp = requests.get(f"{ARCGIS_URL}/query", params=params, timeout=60)
        resp.raise_for_status()
        geojson = resp.json()
        features = geojson.get("features", [])
        if not features:
            break

        for feat in features:
            props = feat.get("properties") or feat.get("attributes", {})
            geom = feat.get("geometry") or {}

            # Resolve date — try common Esri field names
            date_ms = None
            for key in ("OccurrenceDate", "occurrence_date", "date", "Date",
                        "ReportedDate", "reported_date", "REPORT_DATE",
                        "event_date", "EventDate"):
                val = props.get(key)
                if val is not None:
                    date_ms = int(val)
                    break
            if date_ms is None:
                continue  # skip rows with no date
            if date_ms < cutoff_ms:
                continue  # outside lookback window

            coords = geom.get("coordinates", [None, None])
            records.append({
                "objectid": props.get("OBJECTID") or props.get("objectid"),
                "date_ms": date_ms,
                "city": str(props.get("city") or props.get("City") or props.get("CITY") or "unknown"),
                "description": str(
                    props.get("description") or props.get("Description")
                    or props.get("MCI_CATEGORY") or props.get("offence")
                    or props.get("Offence") or props.get("OFFENCE") or "unknown"
                ),
                "case_no": props.get("case_no") or props.get("event_unique_id"),
                "lng": coords[0] if coords and len(coords) > 1 else None,
                "lat": coords[1] if coords and len(coords) > 1 else None,
            })

        if len(features) < page_size:
            break  # last page
        offset += page_size
        print(f"  fetched up to offset {offset} …")

    df = pd.DataFrame(records)
    print(f"fetch_arcgis: {len(df)} rows after cutoff filter")
    return df


df_raw = fetch_arcgis(LOOKBACK_DAYS)

In [ ]:
def _cyclical(val: pd.Series, period: float):
    angle = 2 * math.pi * val / period
    return np.sin(angle), np.cos(angle)


def build_training_table(df: pd.DataFrame, horizons: list[int]) -> pd.DataFrame:
    """Build feature rows for every (city, incident_type, anchor, horizon) combination."""
    df = df.copy()
    df["incident_type"] = df["description"].str.upper().str.strip()

    # Anchor resolution: every 6 hours to keep the notebook fast
    ANCHOR_STEP_MS = 6 * 3600 * 1000
    WARMUP_MS = 8 * 7 * 24 * 3600 * 1000  # 8 weeks
    ONE_MS = 1

    global_min_ms = int(df["date_ms"].min())
    global_max_ms = int(df["date_ms"].max())
    warmup_cutoff_ms = global_min_ms + WARMUP_MS

    # Snap global_min to 6-h boundary
    anchor_start_ms = (
        (warmup_cutoff_ms // ANCHOR_STEP_MS) * ANCHOR_STEP_MS
    )
    anchors_ms = np.arange(anchor_start_ms, global_max_ms, ANCHOR_STEP_MS)

    rows = []

    for (city, itype), group in df.groupby(["city", "incident_type"]):
        ts = group["date_ms"].values.astype(np.int64)
        ts_sorted = np.sort(ts)

        for anchor_ms in anchors_ms:
            anchor_dt = datetime.fromtimestamp(anchor_ms / 1000,
                                               tz=__import__("zoneinfo").ZoneInfo(TIMEZONE))
            dow = anchor_dt.weekday()       # 0=Mon
            hour = anchor_dt.hour
            month = anchor_dt.month

            dow_sin, dow_cos = _cyclical(pd.Series([dow]), 7)
            hour_sin, hour_cos = _cyclical(pd.Series([hour]), 24)
            month_sin, month_cos = _cyclical(pd.Series([month]), 12)

            # Historical windows (before anchor)
            ms_24h = anchor_ms - 24 * 3600 * 1000
            ms_7d  = anchor_ms - 7  * 24 * 3600 * 1000
            ms_4w  = anchor_ms - 4  * 7  * 24 * 3600 * 1000
            ms_8w  = anchor_ms - 8  * 7  * 24 * 3600 * 1000

            def count_in(lo, hi):
                return int(np.searchsorted(ts_sorted, hi, side="right")
                           - np.searchsorted(ts_sorted, lo, side="left"))

            # DOW+hour match within 4w / 8w
            def count_dow_hour(lo_ms):
                mask = (ts_sorted >= lo_ms) & (ts_sorted < anchor_ms)
                if not mask.any():
                    return 0
                subset = ts_sorted[mask]
                dts = [datetime.fromtimestamp(t / 1000,
                            tz=__import__("zoneinfo").ZoneInfo(TIMEZONE))
                       for t in subset]
                return sum(1 for d in dts
                           if d.weekday() == dow and d.hour == hour)

            c4w  = count_dow_hour(ms_4w)
            c8w  = count_dow_hour(ms_8w)
            c24h = count_in(ms_24h, anchor_ms)
            c7d  = count_in(ms_7d,  anchor_ms)

            for horizon_h in horizons:
                horizon_end_ms = anchor_ms + horizon_h * 3600 * 1000
                target = count_in(anchor_ms, horizon_end_ms)

                rows.append({
                    "dow_sin":              float(dow_sin.iloc[0]),
                    "dow_cos":              float(dow_cos.iloc[0]),
                    "hour_sin":             float(hour_sin.iloc[0]),
                    "hour_cos":             float(hour_cos.iloc[0]),
                    "month_sin":            float(month_sin.iloc[0]),
                    "month_cos":            float(month_cos.iloc[0]),
                    "count_4w_same_dow_hour": c4w,
                    "count_8w_same_dow_hour": c8w,
                    "count_24h":            c24h,
                    "count_7d":             c7d,
                    "horizon_hours":        horizon_h,
                    "incident_type":        itype,
                    "city":                 city,
                    "target_count":         target,
                    "anchor_ms":            anchor_ms,
                })

    out = pd.DataFrame(rows)
    print(f"build_training_table: {len(out)} rows")
    return out


df_train_raw = build_training_table(df_raw, HORIZONS_HOURS)

In [ ]:
# ── Train / test split (last 30 days = test) ─────────────────────────────────
split_ms = int(df_train_raw["anchor_ms"].max()) - 30 * 24 * 3600 * 1000
train_mask = df_train_raw["anchor_ms"] <= split_ms
test_mask  = df_train_raw["anchor_ms"] >  split_ms

df_tr = df_train_raw[train_mask].copy()
df_te = df_train_raw[test_mask].copy()
print(f"Train rows: {len(df_tr)}  Test rows: {len(df_te)}")

# ── Label-encode categorical columns ─────────────────────────────────────────
le_type = LabelEncoder()
le_city = LabelEncoder()

all_types = df_train_raw["incident_type"].unique()
all_cities = df_train_raw["city"].unique()
le_type.fit(all_types)
le_city.fit(all_cities)

df_tr["type_id"] = le_type.transform(df_tr["incident_type"])
df_tr["city_id"] = le_city.transform(df_tr["city"])
df_te["type_id"] = le_type.transform(df_te["incident_type"])
df_te["city_id"] = le_city.transform(df_te["city"])

# Persist encoder maps as plain dicts: {label_string: int_id}
type_encoder_map = {cls: int(idx) for idx, cls in enumerate(le_type.classes_)}
city_encoder_map = {cls: int(idx) for idx, cls in enumerate(le_city.classes_)}

FEATURE_NAMES = [
    "dow_sin", "dow_cos",
    "hour_sin", "hour_cos",
    "month_sin", "month_cos",
    "count_4w_same_dow_hour",
    "count_8w_same_dow_hour",
    "count_24h",
    "count_7d",
    "horizon_hours",
    "type_id",
    "city_id",
]

X_train = df_tr[FEATURE_NAMES].values
y_train = df_tr["target_count"].values.astype(float)
X_test  = df_te[FEATURE_NAMES].values
y_test  = df_te["target_count"].values.astype(float)

# ── Train LGBMRegressor ───────────────────────────────────────────────────────
model = lgb.LGBMRegressor(
    objective="poisson",
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=63,
    min_data_in_leaf=20,
    n_jobs=-1,
    verbose=-1,
)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    eval_metric="mae",
    callbacks=[
        lgb.early_stopping(stopping_rounds=30, verbose=True),
        lgb.log_evaluation(period=50),
    ],
)

best_iter = model.best_iteration_
val_mae = model.best_score_["valid_0"]["l1"]
print(f"\nBest iteration: {best_iter}   Validation MAE: {val_mae:.4f}")

In [ ]:
# ── Calibration evaluation on the test set ───────────────────────────────────
y_pred = model.predict(X_test)
y_pred = np.clip(y_pred, 1e-6, None)  # Poisson mu must be > 0

mae = float(np.mean(np.abs(y_pred - y_test)))

# For binary calibration: "actual >= threshold" where threshold = round(mu)
thresholds = np.maximum(np.round(y_pred), 1).astype(int)
p_exceed   = np.array([1 - scipy_poisson.cdf(t - 1, mu=mu)
                        for t, mu in zip(thresholds, y_pred)])
y_binary   = (y_test >= thresholds).astype(int)

brier = float(np.mean((p_exceed - y_binary) ** 2))
log_loss_val = float(sk_log_loss(y_binary, p_exceed, eps=1e-7))

print(f"MAE:       {mae:.4f}")
print(f"Brier:     {brier:.4f}")
print(f"Log-loss:  {log_loss_val:.4f}")

# ── Reliability diagram ───────────────────────────────────────────────────────
n_bins = 10
bin_edges = np.linspace(0, 1, n_bins + 1)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
fraction_pos = []
mean_pred    = []

for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
    mask = (p_exceed >= lo) & (p_exceed < hi)
    if mask.sum() == 0:
        fraction_pos.append(np.nan)
        mean_pred.append((lo + hi) / 2)
    else:
        fraction_pos.append(y_binary[mask].mean())
        mean_pred.append(p_exceed[mask].mean())

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot([0, 1], [0, 1], "k--", label="Perfect calibration")
ax.plot(mean_pred, fraction_pos, "o-", label="Model")
ax.set_xlabel("Mean predicted probability")
ax.set_ylabel("Fraction of positives")
ax.set_title("Reliability diagram")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Export the model snapshot as a self-contained JSON blob ──────────────────
booster = model.booster_
dump    = booster.dump_model()

snapshot = {
    "format":           "lightgbm-treedump-v1",
    "feature_names":    FEATURE_NAMES,
    "categorical": {
        "incident_type": type_encoder_map,
        "city":          city_encoder_map,
    },
    "trees":            dump["tree_info"],
    "init_score":       float(dump.get("average_output", 0.0)),
    "objective":        "poisson",
    "trained_at":       datetime.utcnow().isoformat() + "Z",
    "training_samples": int(len(X_train)),
    "metrics": {
        "mae":      float(mae),
        "brier":    float(brier),
        "log_loss": float(log_loss_val),
    },
    "horizons": HORIZONS_HOURS,
}

snapshot_json = json.dumps(snapshot)
size_mb = len(snapshot_json.encode("utf-8")) / 1_048_576
print(f"Snapshot size: {size_mb:.2f} MB")

# Save to Kaggle working dir if available, otherwise local
out_path = "/kaggle/working/snapshot.json" if os.path.isdir("/kaggle/working") else "./snapshot.json"
with open(out_path, "w") as f:
    f.write(snapshot_json)
print(f"Saved to {out_path}")

In [ ]:
# ── Upload snapshot rows to Supabase ─────────────────────────────────────────
if SUPABASE_URL and SUPABASE_SERVICE_ROLE_KEY:
    endpoint = f"{SUPABASE_URL.rstrip('/')}/rest/v1/prediction_model_snapshots"
    headers = {
        "Authorization":  f"Bearer {SUPABASE_SERVICE_ROLE_KEY}",
        "apikey":         SUPABASE_SERVICE_ROLE_KEY,
        "Content-Type":   "application/json",
        "Prefer":         "resolution=merge-duplicates",
    }
    body = [
        {"model_id": MODEL_ID, "horizon_hours": h, "state": snapshot}
        for h in HORIZONS_HOURS
    ]
    resp = requests.post(endpoint, headers=headers, json=body, timeout=60)
    print(f"Supabase upload status: {resp.status_code}")
    if not resp.ok:
        print(resp.text)
else:
    print("SUPABASE_URL / SUPABASE_SERVICE_ROLE_KEY not set — skipping upload.")

## How to run

### Kaggle

1. Go to **kaggle.com → Code → New Notebook**, then **File → Import Notebook** and upload `train_crime_model.ipynb`.
2. Add your Supabase credentials via **Add-ons → Secrets** (name them `SUPABASE_URL` and `SUPABASE_SERVICE_ROLE_KEY`).
   Read them in the config cell with `os.environ.get("SUPABASE_URL", "")`.
3. Set **Accelerator → None** (CPU is fine for LightGBM) and click **Run All**.
4. The output file will be written to `/kaggle/working/snapshot.json` and auto-persisted as a Kaggle output.

### Google Colab

1. Open **colab.research.google.com**, create a new notebook, and paste the cells from this file (or use **File → Upload notebook**).
2. Set `SUPABASE_URL` and `SUPABASE_SERVICE_ROLE_KEY` directly in the config cell, or read them from Colab Secrets (`userdata.get(...)`).
3. Click **Runtime → Run all**.
4. The snapshot will be saved to `./snapshot.json` in the Colab working directory; download it from the Files panel.

### Node side

Once you click **Upload** (or the Supabase POST completes), the JSON snapshot is stored in the
`prediction_model_snapshots` table keyed by `(model_id, horizon_hours)`.  
The Node side reads it via `getModelStateSnapshot` in
`lib/predictions/infrastructure/supabaseRepos.ts` and the tree-walking evaluator in
`lib/predictions/infrastructure/models/trained.ts` scores each prediction without any ML runtime dependency.